# OpenCV Crash Course: 2027 Edition
## Module 4: Video & Real-time Processing
**Estimated time: 20-25 minutes**

---

### What you'll learn
- Read video files frame by frame
- Apply real-time filters (grayscale, edges, blur)
- Calculate FPS
- Webcam / RTSP camera template - the loop every real-time app uses

> **Note on Jupyter + webcam:** `cv2.imshow()` works best in a standalone `.py` script.  
> In Jupyter, we show frames using **matplotlib** or **IPython display**.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import time

def show(img, title="", figsize=(8, 5)):
    plt.figure(figsize=figsize)
    if len(img.shape) == 3:
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    else:
        plt.imshow(img, cmap='gray')
    plt.title(title, fontsize=13)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

def show_grid(images, titles, cols=3, figsize=(15, 5)):
    rows = (len(images) - 1) // cols + 1
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    axes = np.array(axes).flatten()
    for i, (img, title) in enumerate(zip(images, titles)):
        if len(img.shape) == 3:
            axes[i].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        else:
            axes[i].imshow(img, cmap='gray')
        axes[i].set_title(title, fontsize=11)
        axes[i].axis('off')
    for j in range(len(images), len(axes)):
        axes[j].axis('off')
    plt.tight_layout()
    plt.show()

## 1. Reading Video Files

`VideoCapture` works for:
- Video files: `cv2.VideoCapture('video.mp4')`
- Webcam: `cv2.VideoCapture(0)` (0 = first camera)

In [ ]:
# Synthesize a short video for demo (since we may not have a file)
# This creates a video where a colored ball moves across the frame

import os
os.makedirs('input', exist_ok=True)
out_path = 'input/demo_video.mp4'
fourcc   = cv2.VideoWriter_fourcc(*'mp4v')
fps      = 30
size     = (640, 360)
writer   = cv2.VideoWriter(out_path, fourcc, fps, size)

for f in range(90):  # 3 seconds @ 30fps
    frame = np.zeros((*size[::-1], 3), dtype=np.uint8)
    frame[:] = (20, 20, 40)
    x = int(30 + (size[0] - 60) * f / 90)
    y = int(size[1] // 2 + 80 * np.sin(f * 0.15))
    cv2.circle(frame, (x, y), 25, (0, 100, 255), -1)
    cv2.putText(frame, f"Frame {f:03d}", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (200, 200, 200), 1)
    writer.write(frame)

writer.release()
print(f"Created: {out_path}")

In [ ]:
# Read video properties
cap = cv2.VideoCapture('input/demo_video.mp4')

total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps          = cap.get(cv2.CAP_PROP_FPS)
width        = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height       = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
duration     = total_frames / fps

print(f"Resolution  : {width} x {height}")
print(f"FPS         : {fps}")
print(f"Total frames: {total_frames}")
print(f"Duration    : {duration:.2f} seconds")

cap.release()

## 2. Real-time Processing (Process All Frames)

In a real app you'd use `cv2.imshow()` inside the loop.  
Here we collect processed frames and show a sample strip.

In [ ]:
def process_video(src, filter_fn, label="", sample_every=15):
    """Apply filter_fn to every frame, return sample frames."""
    cap     = cv2.VideoCapture(src)
    samples = []
    idx     = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        processed = filter_fn(frame)
        if idx % sample_every == 0:
            samples.append(processed)
        idx += 1

    cap.release()
    return samples

# Define filter functions
def to_gray(frame):
    g = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    return cv2.cvtColor(g, cv2.COLOR_GRAY2BGR)  # keep 3ch for display

def to_canny(frame):
    g    = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    edge = cv2.Canny(cv2.GaussianBlur(g, (5,5), 0), 50, 150)
    return cv2.cvtColor(edge, cv2.COLOR_GRAY2BGR)

def to_blur(frame):
    return cv2.GaussianBlur(frame, (21, 21), 0)

# Sample frames from each filter
orig_frames  = process_video('input/demo_video.mp4', lambda f: f)
gray_frames  = process_video('input/demo_video.mp4', to_gray)
canny_frames = process_video('input/demo_video.mp4', to_canny)
blur_frames  = process_video('input/demo_video.mp4', to_blur)

print(f"Sampled {len(gray_frames)} frames per filter")

In [ ]:
# Show frame 3 from each filter side by side
show_grid(
    [orig_frames[2], gray_frames[2], canny_frames[2], blur_frames[2]],
    ["Original", "Grayscale", "Canny edges", "Gaussian blur"],
    cols=4, figsize=(16, 4)
)

## 3. FPS Counter

Always measure real-time performance in vision apps.

In [ ]:
# FPS calculation template - use this in any real-time loop
cap         = cv2.VideoCapture('input/demo_video.mp4')
frame_count = 0
fps_values  = []
prev_time   = time.time()

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # --- your processing here ---
    cv2.GaussianBlur(frame, (5, 5), 0)
    # ----------------------------

    curr_time = time.time()
    fps_val   = 1.0 / (curr_time - prev_time + 1e-9)
    fps_values.append(fps_val)
    prev_time = curr_time
    frame_count += 1

cap.release()

avg_fps = np.mean(fps_values[5:])  # skip first few unstable frames
print(f"Processed {frame_count} frames")
print(f"Average FPS: {avg_fps:.1f}")

# In a real app, overlay FPS on the frame:
# cv2.putText(frame, f"FPS: {fps_val:.1f}", (10,30),
#             cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255,0), 2)

## 6. Webcam Capture Template

This is the standard pattern for any **real-time camera app** - run this as a `.py` file.

In [ ]:
# ---------------------------------------------------------------
# WEBCAM TEMPLATE - copy this as a starting point for any
# real-time OpenCV application.
# Run with: python scripts/webcam_template.py
# Press 'q' to quit.
# ---------------------------------------------------------------

webcam_template = '''
import cv2
import time

cap = cv2.VideoCapture(0)   # 0 = default camera
cap.set(cv2.CAP_PROP_FRAME_WIDTH,  1280)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

prev_time = time.time()

while True:
    ret, frame = cap.read()
    if not ret:
        print("Camera not found")
        break

    # ---- YOUR PROCESSING HERE ----
    output = cv2.GaussianBlur(frame, (5, 5), 0)
    # ------------------------------

    # FPS overlay
    curr_time = time.time()
    fps = 1.0 / (curr_time - prev_time + 1e-9)
    prev_time = curr_time
    cv2.putText(output, f"FPS: {fps:.1f}", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

    cv2.imshow("Webcam", output)

    key = cv2.waitKey(1) & 0xFF
    if key == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
'''

os.makedirs('scripts', exist_ok=True)
with open('scripts/webcam_template.py', 'w') as f:
    f.write(webcam_template.strip())

print("Saved: scripts/webcam_template.py")
print("Run:   python scripts/webcam_template.py")

---

## Module 4 Summary

| Task | Code |
|------|------|
| Open video/webcam | `cv2.VideoCapture('file.mp4')` or `cv2.VideoCapture(0)` |
| Open IP / CCTV camera | `cv2.VideoCapture('rtsp://user:pass@ip:554/stream')` |
| Read frame | `ret, frame = cap.read()` |
| Video properties | `cap.get(cv2.CAP_PROP_FPS)` |
| Show frame | `cv2.imshow('title', frame)` |
| Wait key | `cv2.waitKey(1) & 0xFF` |
| Release | `cap.release(); cv2.destroyAllWindows()` |

**Next:** Module 5 - YOLO Object Detection (modern deep-learning CV starts here)